In [ ]:
# -*- coding: utf-8 -*-
"""
feature_engineering_32bits.py

Ingeniería de características sobre el dataset CEAS_08 para generar un
cromosoma binario de 32 genes (bits) por correo, listo para alimentar un
Algoritmo Genético (AG) de detección de phishing/spam.

Cada fila de salida = un individuo del AG (vector de 32 genes 0/1 + label).

Autor: Pos yo :)
"""

import re
from datetime import datetime, timezone

import pandas as pd
from urllib.parse import urlparse


# ---------------------------------------------------------------------------
# 0. UTILIDADES GENERALES
# ---------------------------------------------------------------------------

# Regex para extraer "Nombre Display" y "email" del campo sender/receiver,
# que llega en formato 'Nombre Apellido <usuario@dominio.com>' o solo
# 'usuario@dominio.com'.
EMAIL_REGEX = re.compile(r"([\w\.\-+']+)@([\w\.\-]+)")
DISPLAY_NAME_REGEX = re.compile(r"^\s*\"?([^<\"]+)\"?\s*<")

# Cualquier URL http(s) embebida en el cuerpo del correo.
URL_REGEX = re.compile(r"https?://[^\s\"'<>\)\]]+", re.IGNORECASE)

# Dirección IP en formato dotted-decimal (usada tanto para detectar IP en
# cuerpo como IP directa en una URL).
IP_REGEX = re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b")

# Número de teléfono: secuencias de 7+ dígitos con separadores opcionales
# (+, espacios, guiones, paréntesis). Cubre formatos internacionales simples.
PHONE_REGEX = re.compile(
    r"(\+?\d{1,3}[\s\-.]?)?(\(?\d{2,4}\)?[\s\-.]?){2,4}\d{3,4}"
)

# Dominios de correo gratuito / webmail más comunes (ES + EN, mismos nombres
# en ambos idiomas).
FREE_DOMAINS = [
    "gmail.com", "yahoo.com", "outlook.com", "hotmail.com", "live.com",
    "aol.com", "icloud.com", "protonmail.com", "mail.com",
]

# Marcas frecuentemente suplantadas en campañas de phishing.
BRANDS = [
    "google", "paypal", "bank", "banco", "apple", "microsoft", "amazon",
    "facebook", "netflix", "visa", "mastercard", "bbva", "santander",
]

# Acortadores de URL usados para ocultar el destino real del enlace.
SHORTENERS = ["bit.ly", "tinyurl", "goo.gl", "t.co"]

# Palabras clave sospechosas dentro de la propia URL (path/query), típicas
# de páginas de phishing que imitan flujos de login/verificación.
URL_KEYWORDS = ["login", "verify", "secure", "update", "verificar", "acceso"]


def _build_pattern(keywords: list[str]) -> str:
    """Construye un patrón regex 'kw1|kw2|...' escapando caracteres especiales.
    Se usa con .str.contains(patron, case=False, na=False) para cobertura
    bilingüe (ES+EN) en una sola pasada."""
    return "|".join(re.escape(kw) for kw in keywords)


def _extraer_email_y_dominio(texto: str) -> tuple[str | None, str | None]:
    """Extrae (email_completo, dominio) de un campo sender/receiver."""
    if not isinstance(texto, str):
        return None, None
    match = EMAIL_REGEX.search(texto)
    if not match:
        return None, None
    usuario, dominio = match.group(1), match.group(2)
    return f"{usuario}@{dominio}".lower(), dominio.lower()


def _extraer_display_name(texto: str) -> str | None:
    """Extrae el nombre de display de 'Nombre <email>'. None si no hay."""
    if not isinstance(texto, str):
        return None
    match = DISPLAY_NAME_REGEX.match(texto)
    return match.group(1).strip().lower() if match else None


def _extraer_urls(texto: str) -> list[str]:
    """Lista de URLs http(s) encontradas en el texto."""
    if not isinstance(texto, str):
        return []
    return URL_REGEX.findall(texto)


def _dominio_de_url(url: str) -> str:
    """Dominio (netloc, sin 'www.') de una URL."""
    try:
        netloc = urlparse(url).netloc.lower()
        return netloc.replace("www.", "")
    except Exception:
        return ""


def _parsear_fecha(fecha_str: str) -> datetime | None:
    """Parsea fechas en formato RFC 2822, ej: 'Tue, 05 Aug 2008 16:31:02 -0700'."""
    if not isinstance(fecha_str, str):
        return None
    try:
        # email.utils.parsedate_to_datetime soporta el formato RFC 2822
        from email.utils import parsedate_to_datetime
        return parsedate_to_datetime(fecha_str)
    except Exception:
        return None


# ---------------------------------------------------------------------------
# 1. FUNCIÓN POR FILA: extraer_caracteristicas_32(fila)
# ---------------------------------------------------------------------------
def extraer_caracteristicas_32(fila: pd.Series) -> list[int]:
    """
    Recibe una fila del dataset CEAS_08 (sender, receiver, date, subject,
    body, urls, label) y devuelve una lista de 32 enteros (0/1):
    el cromosoma/vector de genes de ese correo.

    El orden de los genes sigue exactamente el mapeo de bits 0-31 definido
    en la especificación (Remitente, Asunto, Temporal, Cuerpo, URLs).
    """
    sender = fila.get("sender", "")
    receiver = fila.get("receiver", "")
    # Forzamos a string vacío cuando el valor es NaN/float (pandas lee
    # celdas vacías como NaN, no como ""), para que todas las regex
    # reciban siempre un str y no rompan con TypeError.
    subject = fila.get("subject", "")
    subject = "" if pd.isna(subject) else str(subject)
    body = fila.get("body", "")
    body = "" if pd.isna(body) else str(body)
    date_str = fila.get("date", "")

    sender_email, sender_dominio = _extraer_email_y_dominio(sender)
    sender_display = _extraer_display_name(sender)
    receiver_email, receiver_dominio = _extraer_email_y_dominio(receiver)

    urls_en_cuerpo = _extraer_urls(body)
    dominios_url = [_dominio_de_url(u) for u in urls_en_cuerpo]

    genes = [0] * 32

    # =======================================================================
    # BLOQUE 1 — REMITENTE (bits 0-4)
    # =======================================================================

    # Bit 0: Dominio externo -> el dominio del remitente difiere del
    # dominio del destinatario. Señal de que el correo no viene de la
    # propia organización del receptor.
    if sender_dominio and receiver_dominio:
        genes[0] = int(sender_dominio != receiver_dominio)

    # Bit 1: Nombre de display sospechoso -> el nombre mostrado no guarda
    # relación con el dominio del email (ej. "PayPal Security" pero el
    # dominio real es 'randommailer.cn'). Heurística: si el display name
    # contiene una marca conocida que NO aparece en el dominio del email,
    # es mismatch (suplantación vía nombre).
    if sender_display and sender_dominio:
        marca_en_nombre = next(
            (b for b in BRANDS if b in sender_display), None
        )
        if marca_en_nombre and marca_en_nombre not in sender_dominio:
            genes[1] = 1

    # Bit 2: Dominio gratuito (gmail, yahoo, outlook, hotmail, etc.)
    # Los correos corporativos/bancarios legítimos rara vez usan webmail
    # gratuito como dominio remitente.
    if sender_dominio:
        genes[2] = int(any(fd in sender_dominio for fd in FREE_DOMAINS))

    # Bit 3: Suplantación de marca conocida -> el dominio del remitente NO
    # es el dominio oficial de la marca, pero el texto (display name o
    # dominio) menciona una marca conocida (Google, PayPal, bancos, etc.).
    if sender_dominio:
        for marca in BRANDS:
            menciona_marca = (sender_display and marca in sender_display) or (
                marca in sender_dominio
            )
            es_dominio_oficial = sender_dominio == f"{marca}.com"
            if menciona_marca and not es_dominio_oficial:
                genes[3] = 1
                break

    # Bit 4: Destinatario 'Undisclosed' o vacío -> el campo receiver está
    # vacío/NaN o contiene la cadena típica 'undisclosed-recipients'.
    receiver_str = str(receiver) if pd.notna(receiver) else ""
    genes[4] = int(
        receiver_str.strip() == ""
        or "undisclosed" in receiver_str.lower()
    )

    # =======================================================================
    # BLOQUE 2 — ASUNTO (bits 5-10)
    # =======================================================================

    patron_urgencia = _build_pattern(
        ["urgent", "urgente", "immediate", "inmediato", "now", "ahora",
         "hurry", "rápido", "rapido"]
    )
    patron_financiero = _build_pattern(
        ["bank", "banco", "payment", "pago", "account", "cuenta"]
    )
    patron_premio = _build_pattern(
        ["winner", "ganador", "prize", "premio", "lottery", "lotería",
         "loteria"]
    )
    patron_amenaza = _build_pattern(
        ["blocked", "bloqueo", "warning", "suspended", "suspensión",
         "suspension"]
    )
    patron_verificacion = _build_pattern(
        ["verify", "verificar", "confirm", "confirmar", "validar",
         "update"]
    )

    # Bit 5: Urgencia en el asunto (ES+EN).
    genes[5] = int(bool(re.search(patron_urgencia, subject, re.IGNORECASE)))

    # Bit 6: Términos financieros en el asunto (ES+EN).
    genes[6] = int(bool(re.search(patron_financiero, subject, re.IGNORECASE)))

    # Bit 7: Premio/recompensa mencionada en el asunto (ES+EN).
    genes[7] = int(bool(re.search(patron_premio, subject, re.IGNORECASE)))

    # Bit 8: Amenaza o consecuencia negativa en el asunto (ES+EN).
    genes[8] = int(bool(re.search(patron_amenaza, subject, re.IGNORECASE)))

    # Bit 9: Solicitud de verificación/confirmación en el asunto (ES+EN).
    genes[9] = int(bool(re.search(patron_verificacion, subject, re.IGNORECASE)))

    # Bit 10: Asunto anormalmente corto (< 5 caracteres), incluye asunto
    # vacío/NaN.
    genes[10] = int(len(str(subject).strip()) < 5)

    # =======================================================================
    # BLOQUE 3 — TEMPORAL (bits 11-12)
    # =======================================================================

    fecha = _parsear_fecha(date_str)

    # Bit 11: Envío en horario de madrugada (00:00 - 05:00), hora local
    # según el offset declarado en el propio header de fecha. Las campañas
    # automatizadas de spam suelen dispararse en estas franjas.
    if fecha is not None:
        genes[11] = int(0 <= fecha.hour < 5)

    # Bit 12: Zona horaria inconsistente -> offsets poco usuales/no
    # estándar (la mayoría de zonas reales son múltiplos de 30 min, entre
    # -12:00 y +14:00). Un offset fuera de ese rango es señal de header
    # falsificado o mal configurado.
    if fecha is not None and fecha.utcoffset() is not None:
        offset_minutos = fecha.utcoffset().total_seconds() / 60
        genes[12] = int(not (-12 * 60 <= offset_minutos <= 14 * 60)
                         or offset_minutos % 30 != 0)

    # =======================================================================
    # BLOQUE 4 — CUERPO / ESTRUCTURAL (bits 13-25)
    # =======================================================================

    patron_credenciales = _build_pattern(
        ["password", "login", "credenciales", "acceso", "contraseña",
         "contrasena"]
    )
    patron_datos_bancarios = _build_pattern(
        ["credit card", "tarjeta", "cvv", "cuenta", "débito", "debito"]
    )
    patron_miedo = _build_pattern(
        ["fear", "perder", "riesgo", "warning", "risk", "lose"]
    )
    patron_legal = _build_pattern(
        ["court", "legal", "juzgado", "demanda", "lawsuit", "tribunal"]
    )
    patron_confidencialidad = _build_pattern(
        ["do not forward", "no reenviar", "confidential", "confidencial",
         "keep this private"]
    )
    patron_beneficio = _build_pattern(
        ["free", "gratis", "bonus", "reward", "recompensa", "beneficio",
         "claim your", "reclama tu"]
    )

    # Bit 13: Solicita credenciales (password/login/credenciales/acceso).
    genes[13] = int(bool(re.search(patron_credenciales, body, re.IGNORECASE)))

    # Bit 14: Solicita datos bancarios (credit card/tarjeta/cvv/cuenta).
    genes[14] = int(bool(re.search(patron_datos_bancarios, body, re.IGNORECASE)))

    # Bit 15: Errores ortográficos / puntuación inusual -> heurística:
    # alta densidad de signos de puntuación repetidos (!!, ??, ...) o
    # mezcla irregular de mayúsculas dentro de palabras (ej. "FrEE").
    puntuacion_inusual = bool(re.search(r"[!?]{2,}|\.{3,}", body))
    mayus_irregular = bool(re.search(r"\b[A-Za-z]*[a-z][A-Z][A-Za-z]*\b", body))
    genes[15] = int(puntuacion_inusual or mayus_irregular)

    # Bit 16: Mezcla de idiomas detectada -> el cuerpo contiene palabras
    # clave en español Y en inglés simultáneamente (mismo concepto en
    # ambos idiomas), señal de plantillas de phishing mal traducidas o
    # generadas automáticamente.
    tiene_es = bool(re.search(
        r"urgente|banco|cuenta|verificar|premio|gratis", body, re.IGNORECASE
    ))
    tiene_en = bool(re.search(
        r"urgent|bank|account|verify|prize|free", body, re.IGNORECASE
    ))
    genes[16] = int(tiene_es and tiene_en)

    # Bit 17: Fecha límite / cuenta regresiva (deadline, expira, dentro de
    # X horas/días).
    patron_deadline = _build_pattern(
        ["deadline", "expires", "expira", "within 24 hours",
         "dentro de 24 horas", "cuenta regresiva", "countdown"]
    )
    genes[17] = int(bool(re.search(patron_deadline, body, re.IGNORECASE)))

    # Bit 18: Apelación al miedo/pérdida (fear/perder/riesgo/warning).
    genes[18] = int(bool(re.search(patron_miedo, body, re.IGNORECASE)))

    # Bit 19: Número de teléfono detectado en el cuerpo (regex).
    genes[19] = int(bool(PHONE_REGEX.search(body)))

    # Bit 20: Cuerpo corto (< 50 palabras) -> los phishing suelen ser
    # breves para presionar una acción rápida.
    n_palabras = len(str(body).split())
    genes[20] = int(n_palabras < 50)

    # Bit 21: Solo contiene un enlace sin texto relevante -> hay
    # exactamente 1 URL y, quitando esa URL, queda muy poco texto
    # (< 15 palabras), patrón típico de "click-bait" puro.
    if len(urls_en_cuerpo) == 1:
        texto_sin_url = URL_REGEX.sub("", body)
        genes[21] = int(len(texto_sin_url.split()) < 15)

    # Bit 22: Promesa de beneficio/recompensa (free/gratis/bonus/etc.).
    genes[22] = int(bool(re.search(patron_beneficio, body, re.IGNORECASE)))

    # Bit 23: Acción legal mencionada (court/legal/juzgado/demanda).
    genes[23] = int(bool(re.search(patron_legal, body, re.IGNORECASE)))

    # Bit 24: Instrucción de confidencialidad / no reenviar.
    genes[24] = int(bool(re.search(patron_confidencialidad, body, re.IGNORECASE)))

    # Bit 25: Uso de una dirección IP literal dentro del cuerpo (fuera de
    # contexto de URL), señal de infraestructura no asociada a un dominio
    # registrado legítimamente.
    genes[25] = int(bool(IP_REGEX.search(body)))

    # =======================================================================
    # BLOQUE 5 — URLs (bits 26-31)
    # =======================================================================

    # Bit 26: URL acortada (bit.ly, t.co, tinyurl, goo.gl).
    patron_shorteners = _build_pattern(SHORTENERS)
    genes[26] = int(bool(re.search(patron_shorteners, body, re.IGNORECASE)))

    # Bit 27: URL con IP directa en lugar de dominio (ej. http://1.2.3.4/...).
    genes[27] = int(any(IP_REGEX.search(u) for u in urls_en_cuerpo))

    # Bit 28: Mismatch entre el dominio del remitente y el dominio de
    # alguno de los enlaces del cuerpo (suplantación de identidad).
    if sender_dominio and dominios_url:
        genes[28] = int(any(
            sender_dominio not in d and d not in sender_dominio
            for d in dominios_url if d
        ))

    # Bit 29: Protocolo HTTP inseguro (no HTTPS) en al menos un enlace.
    genes[29] = int(bool(re.search(r"http://", body, re.IGNORECASE)))

    # Bit 30: Palabras clave sospechosas dentro de la propia URL (login,
    # verify, secure, update, verificar, acceso).
    patron_url_kw = _build_pattern(URL_KEYWORDS)
    genes[30] = int(any(
        re.search(patron_url_kw, u, re.IGNORECASE) for u in urls_en_cuerpo
    ))

    # Bit 31: Múltiples URLs en el cuerpo (> 3) -> spray de enlaces,
    # común en campañas masivas de phishing/spam.
    genes[31] = int(len(urls_en_cuerpo) > 3)

    return genes


# ---------------------------------------------------------------------------
# 2. TRANSFORMACIÓN DEL DATASET COMPLETO
# ---------------------------------------------------------------------------
# Nombres descriptivos para cada uno de los 32 bits, en el mismo orden en
# que los construye extraer_caracteristicas_32 (bit 0 -> primer nombre,
# bit 31 -> último nombre). Se usan como encabezados del DataFrame final
# en lugar de 'bit_00'...'bit_31', para que cada gen sea legible a simple
# vista al inspeccionar el CSV o depurar el AG.
NOMBRES_BITS = [
    # Remitente (bits 0-4)
    "dominio_externo",
    "nombre_display_sospechoso",
    "dominio_gratuito",
    "suplantacion_marca",
    "destinatario_undisclosed",
    # Asunto (bits 5-10)
    "asunto_urgencia",
    "asunto_financiero",
    "asunto_premio",
    "asunto_amenaza",
    "asunto_verificacion",
    "asunto_corto",
    # Temporal (bits 11-12)
    "envio_madrugada",
    "zona_horaria_inconsistente",
    # Cuerpo / Estructural (bits 13-25)
    "solicita_credenciales",
    "solicita_datos_bancarios",
    "errores_ortograficos",
    "mezcla_idiomas",
    "fecha_limite",
    "apelacion_miedo",
    "telefono_detectado",
    "cuerpo_corto",
    "solo_enlace_sin_texto",
    "promesa_beneficio",
    "accion_legal",
    "instruccion_confidencialidad",
    "ip_en_cuerpo",
    # URLs (bits 26-31)
    "url_acortada",
    "url_con_ip",
    "mismatch_dominio_url",
    "protocolo_http_inseguro",
    "palabras_clave_url",
    "multiples_urls",
]


def transformar_dataset_32bits(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aplica extraer_caracteristicas_32 a cada fila del DataFrame y construye
    el DataFrame final de 33 columnas: 32 genes con nombre descriptivo
    (ver NOMBRES_BITS) + label.
    """
    genes_por_fila = df.apply(extraer_caracteristicas_32, axis=1)
    matriz = pd.DataFrame(
        genes_por_fila.tolist(), columns=NOMBRES_BITS, index=df.index
    )

    matriz["label"] = df["label"].astype(int)
    return matriz


# ---------------------------------------------------------------------------
# 3. EJECUCIÓN / DEMO
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    RUTA_CSV = "CEAS_08.csv"
    RUTA_SALIDA = "dataset_phishing_32bits.csv"

    print(f"Cargando dataset desde '{RUTA_CSV}'...")
    df_raw = pd.read_csv(RUTA_CSV)
    print(f"Filas cargadas: {len(df_raw)}")

    print("Extrayendo vector de 32 bits por correo (puede tardar unos segundos)...")
    df_32bits = transformar_dataset_32bits(df_raw)

    print(f"\nShape final: {df_32bits.shape}  (filas, 32 genes + label)")
    print("\nMatriz resultante (head):")
    print(df_32bits.head())

    print("\nDistribución de la clase (label):")
    print(df_32bits["label"].value_counts())

    print("\nProporción de activación (1's) por bit:")
    print(df_32bits.drop(columns=["label"]).mean().sort_values(ascending=False))

    df_32bits.to_csv(RUTA_SALIDA, index=False)
    print(f"\nDataset de 32 bits guardado en: {RUTA_SALIDA}")
    print("Cada fila = 1 individuo (cromosoma de 32 genes) listo para el AG.")